Since there is a retraining needed for poor accuracy values. So to hold up the FGP masks we need a interactive environment to retain and experiment with diffrent parameters.

In [1]:

import sys

from torch import nn
import copy
import random
import torch
from pathlib import Path

# Notebook is inside .../PruningNAS/PruningNAS, so add project root (.../PruningNAS)
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
	sys.path.insert(0, str(project_root))

# Install local package in editable mode (run once; safe to re-run)
%pip install -e ..


from PruningNAS.DataProcess.DataPreprocessing import get_dataloaders
from PruningNAS.Utills.EvaluatiorUtills import get_model_size, get_sparsity
from PruningNAS.Utills.PrunUtillCP import ChannelPrunner
from PruningNAS.Utills.PrunUtillFGP import FineGrainedPruner
from PruningNAS.Utills.TrainingModulesUtills import TrainingPrunned, evaluate
from PruningNAS.Utills.Utill import print_model
from PruningNAS.Utills.ViewerUtills import plot_accuracy, plot_loss  # Ensure you import your correct model architecture
seed=0
random.seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

Obtaining file:///D:/Sutanu_WorkSpace/PruningNas
Note: you may need to restart the kernel to use updated packages.
Device: cuda


ERROR: file:///D:/Sutanu_WorkSpace/PruningNas does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
d:\Sutanu_WorkSpace\PruningNAS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Set Static Params here:

In [3]:
import os


current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")


Current working directory: d:\Sutanu_WorkSpace\PruningNAS\PruningNAS


In [ ]:

# Initialize the model
basedir=''
path='./dataset/cifar10'
select_model='Resnet-34'
pruning_type='CP'
#model_path='./checkpoint/vgg_mrl_99.51375579833984.pth'
model_path=r'checkpoint\Resnet-34\Resnet-34_cifar_95.720001.pth'
# Load the saved state_dict correctly
model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

model.to(device)

sparsity_dict   = {
    "conv1": 0.3,

    "layer1.0": 0.3,
    "layer1.1": 0.3,
    "layer1.2": 0.2,

    "layer2.0": 0.2,
    "layer2.1": 0.5,
    "layer2.2": 0.5,
    "layer2.3": 0.5,

    "layer3.0": 0.4,
    "layer3.1": 0.6,
    "layer3.2": 0.6,
    "layer3.3": 0.5,
    "layer3.4": 0.6,
    "layer3.5": 0.5,

    "layer4.0": 0.7,
    "layer4.1": 0.9,
    "layer4.2": 0.9
}

# sparsity_dict ={ 
# 'conv1':0.80,
# 'layer1.0.conv1':0.90,
# 'layer1.0.conv2':0.90,
# 'layer1.1.conv1':0.90,
# 'layer1.1.conv2':0.90,
# 'layer1.2.conv1':0.90,
# 'layer1.2.conv2':0.90,
# 'layer2.0.conv1':0.90,
# 'layer2.0.conv2':0.80,
# 'layer2.0.shortcut.0':0.80,
# 'layer2.1.conv1':0.90,
# 'layer2.1.conv2':0.90,
# 'layer2.2.conv1':0.90,
# 'layer2.2.conv2':0.90,
# 'layer2.3.conv1':0.90,
# 'layer2.3.conv2':0.90,
# 'layer3.0.conv1':0.90,
# 'layer3.0.conv2':0.80,
# 'layer3.0.shortcut.0':0.80,
# 'layer3.1.conv1':0.90,
# 'layer3.1.conv2':0.90,
# 'layer3.2.conv1':0.90,
# 'layer3.2.conv2':0.90,
# 'layer3.3.conv1':0.90,
# 'layer3.3.conv2':0.90,
# 'layer3.4.conv1':0.90,
# 'layer3.4.conv2':0.90,
# 'layer3.5.conv1':0.90,
# 'layer3.5.conv2':0.90,
# 'layer4.0.conv1':0.90,
# 'layer4.0.conv2':0.90,
# 'layer4.0.shortcut.0':0.90,
# 'layer4.1.conv1':0.90,
# 'layer4.1.conv2':0.90,
# 'layer4.2.conv1':0.90,
# 'layer4.2.conv2':0.90,
# 'fc':0.90,}


Define experimental params here:

In [5]:
num_finetune_epochs = 300
lr=0.01

In [6]:
train_dataloader,test_dataloader=get_dataloaders(path, batch_size=256 ) # Basemodel
dense_model_accuracy=evaluate(model,test_dataloader)
print('dense_model_accuracy:',dense_model_accuracy)
pruned_model=copy.deepcopy(model)
if pruning_type=='FGP':
    isCallback=True
    pruner = FineGrainedPruner(pruned_model, sparsity_dict)
elif pruning_type=='CP':
    pruned_model=ChannelPrunner(pruned_model, sparsity_dict,select_model)
    pruner=None
    isCallback=False
else:
    print('pruning_type doesn\'t exists')
    exit

print_model(pruned_model)
print(f'The sparsity of each layer becomes')
for name, param in pruned_model.named_parameters():
    print(f'  {name}: {get_sparsity(param):.2f}')

dense_model_size = get_model_size(model, count_nonzero_only=True)
sparse_model_size = get_model_size(pruned_model, count_nonzero_only=True)

print(f"Sparse model has size={sparse_model_size:.2f} MiB = {sparse_model_size / dense_model_size * 100:.2f}% of dense model size")
sparse_model_accuracy,_ = evaluate(pruned_model, test_dataloader)
print(f"Sparse model has accuracy={sparse_model_accuracy:.2f}% before fintuning")

optimizer = torch.optim.SGD(pruned_model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_finetune_epochs)
criterion = nn.CrossEntropyLoss()

pruned_model_accuracy,best_pruned_model,accuracies,train_losses,test_losses=TrainingPrunned(pruned_model,train_dataloader,test_dataloader,criterion, optimizer, pruner,scheduler=scheduler,num_finetune_epochs=num_finetune_epochs,isCallback=isCallback)



d:\Sutanu_WorkSpace\PruningNAS\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


dense_model_accuracy: (95.72000122070312, 0.0)
conv1.weight torch.Size([64, 3, 3, 3])
bn1.weight torch.Size([64])
bn1.bias torch.Size([64])
layer1.0.conv1.weight torch.Size([64, 64, 3, 3])
layer1.0.bn1.weight torch.Size([64])
layer1.0.bn1.bias torch.Size([64])
layer1.0.conv2.weight torch.Size([64, 64, 3, 3])
layer1.0.bn2.weight torch.Size([64])
layer1.0.bn2.bias torch.Size([64])
layer1.1.conv1.weight torch.Size([64, 64, 3, 3])
layer1.1.bn1.weight torch.Size([64])
layer1.1.bn1.bias torch.Size([64])
layer1.1.conv2.weight torch.Size([64, 64, 3, 3])
layer1.1.bn2.weight torch.Size([64])
layer1.1.bn2.bias torch.Size([64])
layer1.2.conv1.weight torch.Size([64, 64, 3, 3])
layer1.2.bn1.weight torch.Size([64])
layer1.2.bn1.bias torch.Size([64])
layer1.2.conv2.weight torch.Size([64, 64, 3, 3])
layer1.2.bn2.weight torch.Size([64])
layer1.2.bn2.bias torch.Size([64])
layer2.0.conv1.weight torch.Size([128, 64, 3, 3])
layer2.0.bn1.weight torch.Size([128])
layer2.0.bn1.bias torch.Size([128])
layer2.0.c

Sparse model has accuracy=11.45% before fintuning
Finetuning Fine-grained Pruned Sparse Model


    Epoch 1 Test accuracy:92.94% / Best Accuracy: 92.94%, train loss: 0.0401, test loss: 0.2564


    Epoch 2 Test accuracy:93.41% / Best Accuracy: 93.41%, train loss: 0.0270, test loss: 0.2401


...

    Epoch 296 Test accuracy:95.42% / Best Accuracy: 95.50%, train loss: 0.0002, test loss: 0.2505


    Epoch 297 Test accuracy:95.40% / Best Accuracy: 95.50%, train loss: 0.0002, test loss: 0.2523


    Epoch 298 Test accuracy:95.40% / Best Accuracy: 95.50%, train loss: 0.0002, test loss: 0.2529


    Epoch 299 Test accuracy:95.43% / Best Accuracy: 95.50%, train loss: 0.0002, test loss: 0.2515


    Epoch 300 Test accuracy:95.40% / Best Accuracy: 95.50%, train loss: 0.0002, test loss: 0.2498


In [13]:
print(pruned_model_accuracy)


95.5


In [ ]:
print(pruned_model_accuracy)

torch.save(best_pruned_model, f'{basedir}/checkpoint/{select_model}/{pruning_type}/{select_model}_cifar_{pruning_type}_{pruned_model_accuracy}.pth')

basedir='.'
titel_append=f'of {pruning_type} based Pruned {select_model.title()} model'
save_path=f'{basedir}/checkpoint/{select_model}/{pruning_type}/{select_model}_cifar_{pruning_type}'

plot_accuracy(accuracies,titel_append=titel_append,save_path=save_path+'_acc.png' )
plot_loss(train_losses,test_losses,titel_append=titel_append,save_path=save_path+'_loss.png')

11.449999809265137
[92.93999481201172, 93.40999603271484, 90.83999633789062, 93.5199966430664, 93.35999298095703, 93.43999481201172, 93.82999420166016, 93.73999786376953, 93.88999938964844, 93.6199951171875, 93.82999420166016, 93.56999969482422, 94.5199966430664, 93.91999816894531, 94.0, 94.31999969482422, 93.93999481201172, 94.1199951171875, 94.2699966430664, 94.68000030517578, 94.47000122070312, 94.25, 94.37999725341797, 94.27999877929688, 94.13999938964844, 94.65999603271484, 94.06999969482422, 93.75999450683594, 93.88999938964844, 94.14999389648438, 94.22000122070312, 94.1199951171875, 93.95999908447266, 93.7699966430664, 94.08999633789062, 94.57999420166016, 94.5199966430664, 94.72000122070312, 94.43999481201172, 94.87999725341797, 94.54000091552734, 94.5, 94.2699966430664, 94.22999572753906, 94.4000015258789, 94.27999877929688, 94.45999908447266, 94.4000015258789, 94.25999450683594, 94.5999984741211, 94.47999572753906, 94.43999481201172, 94.44999694824219, 94.14999389648438, 94.4